<a href="https://colab.research.google.com/github/sayja-yug/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sayja-yug/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 1. Two Paper Findings + My Methodology Questions

## Purpose

This section practices the same kind of methodological thinking that I will
apply to my own Week-5 model.

I am not trying to grade the research paper. Instead, I am identifying two
specific findings and asking what evidence I would need to understand whether
the methodology supports those findings.

My lane is:

**Lane 2 — Refresh / Content Opportunity Scoring**

My Week-5 task was to identify content that may deserve review by predicting
whether observed April clicks would decline relative to March.

The important principle for this audit is:

> A strong metric is only meaningful when the label definition and validation
> design support the claim being made.

---

## Finding 1 — Model performance / predictive result

### Finding

The paper reports a model result and uses evaluation metrics to demonstrate
performance.

### My methodology question

**Where exactly does the label come from, and is the label independently
observed rather than being derived from information that also enters the model?**

I would want to verify:

- how the positive and negative examples were defined;
- whether the label was available only after the prediction window;
- whether any label-derived field was included as a feature;
- whether the label definition matches the real decision being supported;
- whether the evaluation data was kept separate from model development.

### Why this matters for my own model

In my Week-5 model, I defined:

`future_decline = 1`

when April clicks were at least 20% lower than March clicks.

Therefore, April information must only be used to create the observed target.
April clicks, April impressions, April position, or the final decline calculation
must not become model features.

This is an important distinction between:

**predicting a future outcome**

and

**describing an outcome that has already happened.**

My Week-6 audit will explicitly check this boundary.

---

## Finding 2 — Validation / generalization result

### Finding

The paper reports evaluation results intended to show how well the proposed
method performs beyond the data used to develop it.

### My methodology question

**Does the validation design actually test the generalization claim being made?**

I would want to know:

- whether train and validation examples can belong to the same underlying
  client/entity;
- whether related observations can appear on both sides of the split;
- whether the split is grouped or time-aware when the data structure requires it;
- whether preprocessing was fitted only on the training data;
- whether the reported metric is calculated on genuinely held-out data.

### Why this matters for my own model

My FlyRank data contains repeated observations for the same client and content.

A random row-level split could therefore make the task easier than the real
deployment situation because the same client's patterns could appear in both
training and testing.

In Week-5 I already used a client-grouped split.

In Week-6 I will explicitly compare the validation design and verify that the
evaluation remains honest.

---

## What I am taking from the paper

The main lesson I am applying is not simply "use a better model."

The lesson is:

1. Define the label independently.
2. Make the validation design match the claim.
3. Check for leakage before trusting the metric.
4. Inspect real failures instead of relying only on a single score.
5. Rewrite claims when the evidence is weaker than the original wording.

For my Lane-2 project, the model is decision-support for prioritizing content
review. It does not prove that refreshing a page will cause recovery.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 2. My Model Under an Honest Split — Before / After

## Objective

The purpose of this section is to test whether my Week-5 model performance
remains supported when the validation design reflects the structure of the
data.

My Lane-2 decision is:

> Which content pages should be reviewed first because they show evidence of
> future click decline?

The prediction setup is:

**March 2026 features → April 2026 observed outcome**

---

## Feature window

March 2026 is the decision/feature window.

The model can use information known by the end of March:

- March impressions
- March clicks
- March average position

The modelling grain is:

**one row per client × content item**

---

## Future outcome

April 2026 is the future outcome window.

The target is:

`future_decline = 1`

when April clicks are at least 20% lower than March clicks.

April information is used only to create the observed target.

It must never be used as a model feature.

---

## Baseline

The Week-4 `baseline_score` is used only as the comparison system.

It is NOT a feature of the machine-learning model.

This prevents the model from simply learning the previous decision rule.

---

## Before validation design

The Week-5 model used a client-grouped train/test split.

This means complete clients are assigned to either training or testing,
rather than allowing the same client to appear in both.

This is preferable to a simple random row split because the data contains
many observations belonging to the same client.

---

## After / audit design

For this validation audit, I will verify the grouped split explicitly.

The key condition is:

**No client may appear in both training and testing.**

If the same client appears in both groups, the validation result may be
optimistic because the model can benefit from client-specific patterns that
it has already seen.

---

## What I will compare

I will compare:

1. Week-4 baseline
2. Week-5 Logistic Regression
3. Week-5 Random Forest

using the same held-out test rows and the same target.

The primary ranking metrics are:

- Precision@100
- Precision@500
- Precision@1000
- Average Precision

These metrics are more relevant than accuracy because the actual use case is
a ranked review queue rather than classification of every page equally.

---

## What makes the validation honest?

An honest validation requires:

- no future April features;
- no `future_decline` feature;
- no `baseline_score` feature;
- no `reason_code` feature;
- no `action_label` feature;
- no client overlap between training and testing;
- evaluation on held-out data;
- comparison against the Week-4 baseline on the same test population.

The goal is not to make the model look better.

The goal is to determine whether the observed improvement survives a
validation design that matches the real decision problem.

In [13]:
# ============================================================
# WEEK 6 — SECTION 2
# HONEST VALIDATION AUDIT
#
# Lane 2: Refresh / Content Opportunity Scoring
#
# Data source:
# week5_comparison_df.csv
#
# Important:
# - No Hugging Face import
# - No datasets package
# - No pyarrow
# - Week-4 baseline is NOT a model feature
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 75)
print("WEEK 6 — SECTION 2: HONEST VALIDATION AUDIT")
print("=" * 75)


# ============================================================
# 1. Locate the Week-5 comparison dataset
# ============================================================

possible_paths = [
    "/content/week5_comparison_df.csv",
    "/content/work/outputs/week5_comparison_df.csv",
    "week5_comparison_df.csv",
    "work/outputs/week5_comparison_df.csv"
]

comparison_path = None

for path in possible_paths:
    if os.path.exists(path):
        comparison_path = path
        break


if comparison_path is None:

    print("\n❌ Week-5 comparison dataset was not found.")

    print("\nFiles currently visible in /content:")

    try:
        for filename in sorted(os.listdir("/content")):
            print(" ", filename)
    except Exception:
        pass

    raise FileNotFoundError(
        """
Upload week5_comparison_df.csv into the Week-6 Colab notebook.

This file must contain:

client_hash_id
content_hash_id
march_impressions
march_clicks
march_avg_position
future_decline
baseline_score

The Week-4 baseline CSV alone cannot create future_decline
because it contains March data but not the observed April outcome.
"""
    )


print("\n✅ Week-5 comparison dataset found:")
print(comparison_path)


# ============================================================
# 2. Load comparison dataset
# ============================================================

comparison_df = pd.read_csv(comparison_path)

print("\n" + "=" * 75)
print("COMPARISON DATASET LOADED")
print("=" * 75)

print("Shape:", comparison_df.shape)

print("\nColumns:")
print(comparison_df.columns.tolist())


# ============================================================
# 3. Required columns
# ============================================================

required_columns = [
    "client_hash_id",
    "content_hash_id",
    "march_impressions",
    "march_clicks",
    "march_avg_position",
    "future_decline",
    "baseline_score"
]

missing_columns = [
    col
    for col in required_columns
    if col not in comparison_df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + str(missing_columns)
    )

print("\n✅ All required columns are present.")


# ============================================================
# 4. Keep only the required audit columns
# ============================================================

comparison_df = comparison_df[
    required_columns
].copy()


# ============================================================
# 5. Remove invalid rows
# ============================================================

feature_columns = [
    "march_impressions",
    "march_clicks",
    "march_avg_position"
]

comparison_df = comparison_df.dropna(
    subset=feature_columns + ["future_decline", "baseline_score"]
).copy()

comparison_df["future_decline"] = (
    comparison_df["future_decline"]
    .astype(int)
)

print("\nRows after removing missing model values:")
print(len(comparison_df))


# ============================================================
# 6. Check target
# ============================================================

print("\n" + "=" * 75)
print("TARGET CHECK")
print("=" * 75)

print(
    comparison_df["future_decline"]
    .value_counts()
    .sort_index()
)

print("\nTarget rate:")

target_rate = comparison_df["future_decline"].mean()

print(f"{target_rate:.3%}")


# ============================================================
# 7. Define X and y
# ============================================================

X = comparison_df[
    feature_columns
].copy()

y = comparison_df[
    "future_decline"
].copy()

groups = comparison_df[
    "client_hash_id"
].copy()


print("\n" + "=" * 75)
print("MODEL OBJECTS")
print("=" * 75)

print("Features:")
print(feature_columns)

print("\nTarget: future_decline")

print(
    "\nUnique clients:",
    comparison_df["client_hash_id"].nunique()
)


# ============================================================
# 8. BEFORE — conventional random split
# ============================================================

print("\n" + "=" * 75)
print("BEFORE: RANDOM TRAIN / TEST SPLIT")
print("=" * 75)

X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=42,
        stratify=y
    )
)


# ============================================================
# 9. Train Logistic Regression on random split
# ============================================================

random_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

random_model.fit(
    X_train_random,
    y_train_random
)

random_probability = random_model.predict_proba(
    X_test_random
)[:, 1]

random_prediction = (
    random_probability >= 0.50
).astype(int)


random_ap = average_precision_score(
    y_test_random,
    random_probability
)

random_precision = precision_score(
    y_test_random,
    random_prediction,
    zero_division=0
)

random_recall = recall_score(
    y_test_random,
    random_prediction,
    zero_division=0
)

random_f1 = f1_score(
    y_test_random,
    random_prediction,
    zero_division=0
)


# ============================================================
# 10. AFTER — client-grouped split
# ============================================================

print("\n" + "=" * 75)
print("AFTER: CLIENT-GROUPED TRAIN / TEST SPLIT")
print("=" * 75)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups=groups
    )
)

X_train_grouped = X.iloc[train_idx].copy()
X_test_grouped = X.iloc[test_idx].copy()

y_train_grouped = y.iloc[train_idx].copy()
y_test_grouped = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]


# ============================================================
# 11. Verify NO client overlap
# ============================================================

train_clients = set(
    groups_train.unique()
)

test_clients = set(
    groups_test.unique()
)

client_overlap = (
    train_clients.intersection(
        test_clients
    )
)

print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Client overlap:", len(client_overlap))

if len(client_overlap) != 0:
    raise RuntimeError(
        "Client leakage detected: "
        "some clients occur in both train and test."
    )

print("✅ No client overlap.")


# ============================================================
# 12. Train Logistic Regression on grouped split
# ============================================================

grouped_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_probability = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_prediction = (
    grouped_probability >= 0.50
).astype(int)


grouped_ap = average_precision_score(
    y_test_grouped,
    grouped_probability
)

grouped_precision = precision_score(
    y_test_grouped,
    grouped_prediction,
    zero_division=0
)

grouped_recall = recall_score(
    y_test_grouped,
    grouped_prediction,
    zero_division=0
)

grouped_f1 = f1_score(
    y_test_grouped,
    grouped_prediction,
    zero_division=0
)


# ============================================================
# 13. Evaluate Week-4 baseline on SAME grouped test rows
# ============================================================

baseline_test = comparison_df.iloc[
    test_idx
]["baseline_score"].values

baseline_ap = average_precision_score(
    y_test_grouped,
    baseline_test
)

baseline_prediction = (
    baseline_test >= np.median(baseline_test)
).astype(int)

baseline_precision = precision_score(
    y_test_grouped,
    baseline_prediction,
    zero_division=0
)

baseline_recall = recall_score(
    y_test_grouped,
    baseline_prediction,
    zero_division=0
)

baseline_f1 = f1_score(
    y_test_grouped,
    baseline_prediction,
    zero_division=0
)


# ============================================================
# 14. BEFORE vs AFTER comparison
# ============================================================

results_df = pd.DataFrame({
    "system": [
        "Week-4 Baseline",
        "Logistic Regression - Random Split",
        "Logistic Regression - Grouped Split"
    ],

    "average_precision": [
        baseline_ap,
        random_ap,
        grouped_ap
    ],

    "precision": [
        baseline_precision,
        random_precision,
        grouped_precision
    ],

    "recall": [
        baseline_recall,
        random_recall,
        grouped_recall
    ],

    "f1": [
        baseline_f1,
        random_f1,
        grouped_f1
    ]
})


print("\n" + "=" * 75)
print("BEFORE / AFTER VALIDATION COMPARISON")
print("=" * 75)

display(
    results_df.round(4)
)


# ============================================================
# 15. Calculate validation change
# ============================================================

ap_change = (
    grouped_ap - random_ap
)

print("\n" + "=" * 75)
print("VALIDATION DESIGN CHANGE")
print("=" * 75)

print(
    f"Random-split Average Precision : "
    f"{random_ap:.4f}"
)

print(
    f"Grouped-split Average Precision : "
    f"{grouped_ap:.4f}"
)

print(
    f"Change after grouped validation : "
    f"{ap_change:+.4f}"
)


# ============================================================
# 16. Final audit checks
# ============================================================

print("\n" + "=" * 75)
print("SECTION 2 AUDIT CHECKS")
print("=" * 75)

# No future features
forbidden_features = [
    "april_impressions",
    "april_clicks",
    "april_avg_position",
    "click_change_pct",
    "future_decline",
    "baseline_score",
    "reason_code",
    "action_label"
]

leaked_features = [
    col
    for col in feature_columns
    if col in forbidden_features
]

if leaked_features:
    raise RuntimeError(
        "Potential leakage in model features: "
        + str(leaked_features)
    )

print("✅ Model features contain no future-window variables.")
print("✅ Week-4 baseline is not a model feature.")
print("✅ Future_decline is used only as the observed target.")
print("✅ Random split evaluated.")
print("✅ Client-grouped split evaluated.")
print("✅ Train/test client overlap checked.")
print("✅ Baseline evaluated on the same grouped test rows.")


# ============================================================
# 17. Save results
# ============================================================

os.makedirs(
    "/content/work/outputs",
    exist_ok=True
)

results_path = (
    "/content/work/outputs/"
    "week6_validation_comparison.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

print("\nResults saved to:")
print(results_path)


# ============================================================
# 18. Final status
# ============================================================

print("\n" + "=" * 75)
print("SECTION 2 COMPLETE")
print("=" * 75)

print(
    "Decision / feature window : March 2026"
)

print(
    "Observed outcome window   : April 2026"
)

print(
    "Unit of analysis          : client × content"
)

print(
    "Target                    : future_decline"
)

print(
    "Model features            : "
    + ", ".join(feature_columns)
)

print(
    "Honest validation         : client-grouped split"
)

print(
    "\n✅ Section 2 honest validation audit completed."
)

WEEK 6 — SECTION 2: HONEST VALIDATION AUDIT

✅ Week-5 comparison dataset found:
/content/week5_comparison_df.csv

COMPARISON DATASET LOADED
Shape: (3292, 13)

Columns:
['client_hash_id', 'content_hash_id', 'march_impressions', 'march_clicks', 'march_avg_position', 'april_impressions', 'april_clicks', 'april_avg_position', 'click_change_pct', 'future_decline', 'baseline_score', 'reason_code', 'action_label']

✅ All required columns are present.

Rows after removing missing model values:
3292

TARGET CHECK
future_decline
0    1526
1    1766
Name: count, dtype: int64

Target rate:
53.645%

MODEL OBJECTS
Features:
['march_impressions', 'march_clicks', 'march_avg_position']

Target: future_decline

Unique clients: 22

BEFORE: RANDOM TRAIN / TEST SPLIT

AFTER: CLIENT-GROUPED TRAIN / TEST SPLIT
Training clients: 15
Testing clients: 7
Client overlap: 0
✅ No client overlap.

BEFORE / AFTER VALIDATION COMPARISON


,system,average_precision,precision,recall,f1
0,Week-4 Baseline,0.5564,0.5739,0.4371,0.4962
1,Logistic Regression - Random Split,0.6085,0.5413,0.8660,0.6662
2,Logistic Regression - Grouped Split,0.5754,0.6383,0.7947,0.7080



VALIDATION DESIGN CHANGE
Random-split Average Precision : 0.6085
Grouped-split Average Precision : 0.5754
Change after grouped validation : -0.0331

SECTION 2 AUDIT CHECKS
✅ Model features contain no future-window variables.
✅ Week-4 baseline is not a model feature.
✅ Future_decline is used only as the observed target.
✅ Random split evaluated.
✅ Client-grouped split evaluated.
✅ Train/test client overlap checked.
✅ Baseline evaluated on the same grouped test rows.

Results saved to:
/content/work/outputs/week6_validation_comparison.csv

SECTION 2 COMPLETE
Decision / feature window : March 2026
Observed outcome window   : April 2026
Unit of analysis          : client × content
Target                    : future_decline
Model features            : march_impressions, march_clicks, march_avg_position
Honest validation         : client-grouped split

✅ Section 2 honest validation audit completed.


# 3. Leakage Audit and Real Failure Examples

## Leakage audit

The purpose of this section is to verify that the Week-5 model learned only from information that would have been available at the decision time.

My decision/feature window is March 2026 and my observed outcome window is April 2026.

The model uses only three features:

- `march_impressions`
- `march_clicks`
- `march_avg_position`

These measurements belong to the March feature window and therefore are available before the April outcome is observed.

The following variables are explicitly excluded from model features:

- `april_impressions`
- `april_clicks`
- `april_avg_position`
- `click_change_pct`
- `future_decline`
- `baseline_score`
- `reason_code`
- `action_label`

The April variables and `click_change_pct` are derived from the future outcome window. `future_decline` is the observed target itself. The Week-4 baseline outputs are retained only for comparison and are not model inputs.

The audit therefore checks both:

1. **Future-window leakage:** whether any April or target-derived variable entered the model.
2. **Decision-output leakage:** whether the Week-4 baseline score, reason code, or action label entered the model.

A clean audit does not prove that the model is correct. It only supports the narrower claim that the tested model features respect the information available at decision time.

## Real failure examples

After checking leakage, I inspect false positives and false negatives from the client-grouped test set.

A false positive means the model ranked an item as likely to decline, but the observed April outcome did not meet the decline definition.

A false negative means the model gave a lower risk score even though the item subsequently satisfied the future-decline definition.

These examples are important because aggregate metrics do not explain why individual predictions fail.

I will therefore inspect representative high-confidence false positives and false negatives and use them to identify possible sources of model weakness such as noisy behavior, limited features, client-specific patterns, or threshold effects.

These examples are observational failure cases. They do not prove that the model caused or could have prevented the observed outcomes.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# WEEK 6 — SECTION 3
# LEAKAGE AUDIT + REAL FAILURE EXAMPLES
#
# Lane 2: Refresh / Content Opportunity Scoring
#
# This section audits:
# 1. Model feature provenance
# 2. Future-window leakage
# 3. Week-4 decision-output leakage
# 4. Actual model failures on grouped test data
# ============================================================

import numpy as np
import pandas as pd

print("=" * 75)
print("WEEK 6 — SECTION 3: LEAKAGE AUDIT")
print("=" * 75)


# ============================================================
# 1. Verify required objects from Section 2
# ============================================================

required_objects = [
    "comparison_df",
    "grouped_model",
    "grouped_probability",
    "grouped_prediction",
    "test_idx",
    "y_test_grouped",
    "X_test_grouped"
]

missing_objects = [
    obj
    for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Section 3 cannot run because these Section-2 objects "
        "are missing:\n"
        + str(missing_objects)
        + "\n\nRun Section 2 first."
    )

print("✅ Section-2 objects found.")


# ============================================================
# 2. Define the actual model features
# ============================================================

model_features = [
    "march_impressions",
    "march_clicks",
    "march_avg_position"
]

print("\n" + "=" * 75)
print("ACTUAL MODEL FEATURES")
print("=" * 75)

for feature in model_features:
    print("✅", feature)


# ============================================================
# 3. Define variables that must never be model features
# ============================================================

future_or_target_variables = [
    "april_impressions",
    "april_clicks",
    "april_avg_position",
    "click_change_pct",
    "future_decline"
]

decision_output_variables = [
    "baseline_score",
    "reason_code",
    "action_label"
]

forbidden_variables = (
    future_or_target_variables
    + decision_output_variables
)


# ============================================================
# 4. Leakage audit table
# ============================================================

audit_rows = []

for column in comparison_df.columns:

    if column in model_features:

        role = "MODEL FEATURE"
        future_information = "NO"
        allowed = "YES"

    elif column in future_or_target_variables:

        role = "FUTURE / TARGET-DERIVED"
        future_information = "YES"
        allowed = "NO"

    elif column in decision_output_variables:

        role = "WEEK-4 DECISION OUTPUT"
        future_information = "NOT A MODEL INPUT"
        allowed = "NO"

    elif column in [
        "client_hash_id",
        "content_hash_id"
    ]:

        role = "IDENTIFIER"
        future_information = "NO"
        allowed = "NO"

    else:

        role = "UNCLASSIFIED"
        future_information = "UNKNOWN"
        allowed = "REVIEW"


    audit_rows.append({
        "column": column,
        "role": role,
        "future_information": future_information,
        "allowed_as_model_feature": allowed
    })


leakage_audit_df = pd.DataFrame(
    audit_rows
)

print("\n" + "=" * 75)
print("FEATURE / DATA LINEAGE AUDIT")
print("=" * 75)

display(leakage_audit_df)


# ============================================================
# 5. Verify that the actual X matrix contains ONLY
#    the three approved March features
# ============================================================

actual_X_columns = list(
    X_test_grouped.columns
)

print("\n" + "=" * 75)
print("MODEL MATRIX CHECK")
print("=" * 75)

print("Columns actually passed to the model:")
print(actual_X_columns)

unexpected_model_features = [
    col
    for col in actual_X_columns
    if col not in model_features
]

missing_model_features = [
    col
    for col in model_features
    if col not in actual_X_columns
]


if unexpected_model_features:

    raise RuntimeError(
        "❌ Unexpected feature(s) detected in the model matrix: "
        + str(unexpected_model_features)
    )


if missing_model_features:

    raise RuntimeError(
        "❌ Expected model feature(s) missing from model matrix: "
        + str(missing_model_features)
    )


print("✅ Model matrix contains exactly the approved March features.")


# ============================================================
# 6. Explicit future leakage check
# ============================================================

future_used = [
    col
    for col in actual_X_columns
    if col in future_or_target_variables
]

if future_used:

    raise RuntimeError(
        "❌ FUTURE LEAKAGE DETECTED: "
        + str(future_used)
    )

print(
    "✅ No April/future-window variable is present "
    "in the model matrix."
)


# ============================================================
# 7. Explicit target leakage check
# ============================================================

target_used = [
    col
    for col in actual_X_columns
    if col == "future_decline"
]

if target_used:

    raise RuntimeError(
        "❌ TARGET LEAKAGE DETECTED: future_decline "
        "was used as a model feature."
    )

print(
    "✅ future_decline is used only as the observed target."
)


# ============================================================
# 8. Explicit Week-4 decision-output leakage check
# ============================================================

decision_output_used = [
    col
    for col in actual_X_columns
    if col in decision_output_variables
]

if decision_output_used:

    raise RuntimeError(
        "❌ DECISION-OUTPUT LEAKAGE DETECTED: "
        + str(decision_output_used)
    )

print(
    "✅ Week-4 baseline_score/reason_code/action_label "
    "are not model features."
)


# ============================================================
# 9. Check that model features are numeric
# ============================================================

print("\n" + "=" * 75)
print("NUMERIC FEATURE CHECK")
print("=" * 75)

feature_dtypes = (
    X_test_grouped[model_features]
    .dtypes
    .to_frame("dtype")
)

display(feature_dtypes)

non_numeric = [
    col
    for col in model_features
    if not pd.api.types.is_numeric_dtype(
        X_test_grouped[col]
    )
]

if non_numeric:

    raise TypeError(
        "Non-numeric model features found: "
        + str(non_numeric)
    )

print("✅ All model features are numeric.")


# ============================================================
# 10. Check for invalid infinite values
# ============================================================

infinite_counts = (
    np.isinf(
        X_test_grouped[model_features]
        .astype(float)
    )
    .sum()
)

print("\nInfinite values by feature:")
print(infinite_counts)

if infinite_counts.sum() > 0:

    raise ValueError(
        "Infinite values found in model features."
    )

print("✅ No infinite feature values.")


# ============================================================
# 11. Reconstruct grouped test dataframe
# ============================================================

test_df = comparison_df.iloc[
    test_idx
].copy()

test_df = test_df.reset_index(
    drop=True
)

test_df["model_probability"] = (
    np.asarray(grouped_probability)
)

test_df["model_prediction"] = (
    np.asarray(grouped_prediction)
)

test_df["actual_target"] = (
    np.asarray(y_test_grouped)
)


# ============================================================
# 12. Classify prediction errors
# ============================================================

test_df["error_type"] = np.select(
    [
        (
            (test_df["model_prediction"] == 1)
            &
            (test_df["actual_target"] == 1)
        ),

        (
            (test_df["model_prediction"] == 0)
            &
            (test_df["actual_target"] == 0)
        ),

        (
            (test_df["model_prediction"] == 1)
            &
            (test_df["actual_target"] == 0)
        ),

        (
            (test_df["model_prediction"] == 0)
            &
            (test_df["actual_target"] == 1)
        )
    ],
    [
        "Correct Positive",
        "Correct Negative",
        "False Positive",
        "False Negative"
    ],
    default="Unknown"
)


# ============================================================
# 13. Error summary
# ============================================================

print("\n" + "=" * 75)
print("GROUPED TEST ERROR SUMMARY")
print("=" * 75)

error_summary = (
    test_df["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .to_frame("n")
)

error_summary["percentage"] = (
    error_summary["n"]
    / len(test_df)
    * 100
)

display(
    error_summary.round(2)
)


# ============================================================
# 14. False positives
# ============================================================

false_positive_df = (
    test_df[
        test_df["error_type"]
        == "False Positive"
    ]
    .sort_values(
        "model_probability",
        ascending=False
    )
    .copy()
)

print("\n" + "=" * 75)
print("FALSE POSITIVES")
print("=" * 75)

print(
    "False positives:",
    len(false_positive_df)
)

display(
    false_positive_df[
        [
            "client_hash_id",
            "content_hash_id",
            "march_impressions",
            "march_clicks",
            "march_avg_position",
            "model_probability",
            "baseline_score",
            "actual_target"
        ]
    ].head(10)
)


# ============================================================
# 15. False negatives
# ============================================================

false_negative_df = (
    test_df[
        test_df["error_type"]
        == "False Negative"
    ]
    .sort_values(
        "model_probability",
        ascending=False
    )
    .copy()
)

print("\n" + "=" * 75)
print("FALSE NEGATIVES")
print("=" * 75)

print(
    "False negatives:",
    len(false_negative_df)
)

display(
    false_negative_df[
        [
            "client_hash_id",
            "content_hash_id",
            "march_impressions",
            "march_clicks",
            "march_avg_position",
            "model_probability",
            "baseline_score",
            "actual_target"
        ]
    ].head(10)
)


# ============================================================
# 16. High-confidence false positives
# ============================================================

high_conf_fp = (
    false_positive_df[
        false_positive_df["model_probability"] >= 0.80
    ]
    .copy()
)

print("\n" + "=" * 75)
print("HIGH-CONFIDENCE FALSE POSITIVES")
print("=" * 75)

print(
    "High-confidence false positives:",
    len(high_conf_fp)
)

if len(high_conf_fp) > 0:

    display(
        high_conf_fp[
            [
                "client_hash_id",
                "content_hash_id",
                "march_impressions",
                "march_clicks",
                "march_avg_position",
                "model_probability",
                "baseline_score",
                "actual_target"
            ]
        ].head(10)
    )

else:

    print(
        "No false positives with probability >= 0.80."
    )


# ============================================================
# 17. High-confidence false negatives
# ============================================================

high_conf_fn = (
    false_negative_df[
        false_negative_df["model_probability"] <= 0.20
    ]
    .copy()
)

print("\n" + "=" * 75)
print("HIGH-CONFIDENCE FALSE NEGATIVES")
print("=" * 75)

print(
    "High-confidence false negatives:",
    len(high_conf_fn)
)

if len(high_conf_fn) > 0:

    display(
        high_conf_fn[
            [
                "client_hash_id",
                "content_hash_id",
                "march_impressions",
                "march_clicks",
                "march_avg_position",
                "model_probability",
                "baseline_score",
                "actual_target"
            ]
        ].head(10)
    )

else:

    print(
        "No false negatives with probability <= 0.20."
    )


# ============================================================
# 18. Compare model vs baseline on the grouped test set
# ============================================================

print("\n" + "=" * 75)
print("MODEL VS BASELINE — GROUPED TEST SET")
print("=" * 75)

comparison_summary = pd.DataFrame({
    "system": [
        "Week-4 Baseline",
        "Week-5 Logistic Regression"
    ],

    "average_precision": [
        average_precision_score(
            test_df["actual_target"],
            test_df["baseline_score"]
        ),
        average_precision_score(
            test_df["actual_target"],
            test_df["model_probability"]
        )
    ]
})

display(
    comparison_summary.round(4)
)


# ============================================================
# 19. Save leakage audit and errors
# ============================================================

import os

os.makedirs(
    "/content/work/outputs",
    exist_ok=True
)

leakage_audit_path = (
    "/content/work/outputs/"
    "week6_leakage_audit.csv"
)

error_examples_path = (
    "/content/work/outputs/"
    "week6_error_examples.csv"
)

leakage_audit_df.to_csv(
    leakage_audit_path,
    index=False
)

test_df[
    [
        "client_hash_id",
        "content_hash_id",
        "march_impressions",
        "march_clicks",
        "march_avg_position",
        "future_decline",
        "baseline_score",
        "model_probability",
        "model_prediction",
        "error_type"
    ]
].to_csv(
    error_examples_path,
    index=False
)


# ============================================================
# 20. Final status
# ============================================================

print("\n" + "=" * 75)
print("SECTION 3 FINAL STATUS")
print("=" * 75)

print("✅ Model features audited.")
print("✅ Future-window variables checked.")
print("✅ Target leakage checked.")
print("✅ Week-4 decision outputs checked.")
print("✅ Numeric feature validity checked.")
print("✅ Grouped test errors inspected.")
print("✅ False positives identified.")
print("✅ False negatives identified.")

print("\nLeakage audit saved to:")
print(leakage_audit_path)

print("\nError examples saved to:")
print(error_examples_path)

print("\n" + "=" * 75)
print("SECTION 3 COMPLETE")
print("=" * 75)

print(
    "\nNext:"
    "\nSECTION 4 — CLAIM REWRITE"
)

WEEK 6 — SECTION 3: LEAKAGE AUDIT
✅ Section-2 objects found.

ACTUAL MODEL FEATURES
✅ march_impressions
✅ march_clicks
✅ march_avg_position

FEATURE / DATA LINEAGE AUDIT


,column,role,future_information,allowed_as_model_feature
0,client_hash_id,IDENTIFIER,NO,NO
1,content_hash_id,IDENTIFIER,NO,NO
2,march_impressions,MODEL FEATURE,NO,YES
3,march_clicks,MODEL FEATURE,NO,YES
4,march_avg_position,MODEL FEATURE,NO,YES
5,future_decline,FUTURE / TARGET-DERIVED,YES,NO
6,baseline_score,WEEK-4 DECISION OUTPUT,NOT A MODEL INPUT,NO



MODEL MATRIX CHECK
Columns actually passed to the model:
['march_impressions', 'march_clicks', 'march_avg_position']
✅ Model matrix contains exactly the approved March features.
✅ No April/future-window variable is present in the model matrix.
✅ future_decline is used only as the observed target.
✅ Week-4 baseline_score/reason_code/action_label are not model features.

NUMERIC FEATURE CHECK


,dtype
march_impressions,int64
march_clicks,int64
march_avg_position,float64


✅ All model features are numeric.

Infinite values by feature:
march_impressions     0
march_clicks          0
march_avg_position    0
dtype: int64
✅ No infinite feature values.

GROUPED TEST ERROR SUMMARY


...


FALSE NEGATIVES
False negatives: 31


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position,model_probability,baseline_score,actual_target
128,client_fef1a8f436438636,content_25b7a15de19b92d3,3257,6,34.345951,0.499643,0.035952,1
149,client_fef1a8f436438636,content_53c0a2a7e7f04b0e,6210,6,33.806916,0.499420,0.037701,1
178,client_fef1a8f436438636,content_91487e6f59e8e82f,16428,15,31.530447,0.498802,0.072680,1
132,client_fef1a8f436438636,content_361e795eecc3e6ae,5039,6,34.439654,0.497492,0.036619,1
170,client_fef1a8f436438636,content_81707f944166e061,25075,16,31.081167,0.492514,0.047638,1
211,client_fef1a8f436438636,content_d681f96121e7867c,6145,35,34.607047,0.489755,0.037971,1
188,client_fef1a8f436438636,content_a4b7a08e7ba7ce17,2461,5,36.630236,0.489615,0.037702,1
141,client_fef1a8f436438636,content_4bd3ae79787e536f,4147,15,36.065267,0.488699,0.042175,1
166,client_fef1a8f436438636,content_7d9eaf5d66bc0380,2272,5,37.049476,0.487779,0.041537,1
114,client_fef1a8f436438636,content_0c4bfb897ad619da,18899,14,33.772069,0.485866,0.044986,1



HIGH-CONFIDENCE FALSE POSITIVES
High-confidence false positives: 0
No false positives with probability >= 0.80.

HIGH-CONFIDENCE FALSE NEGATIVES
High-confidence false negatives: 0
No false negatives with probability <= 0.20.

MODEL VS BASELINE — GROUPED TEST SET


,system,average_precision
0,Week-4 Baseline,0.5564
1,Week-5 Logistic Regression,0.5754



SECTION 3 FINAL STATUS
✅ Model features audited.
✅ Future-window variables checked.
✅ Target leakage checked.
✅ Week-4 decision outputs checked.
✅ Numeric feature validity checked.
✅ Grouped test errors inspected.
✅ False positives identified.
✅ False negatives identified.

Leakage audit saved to:
/content/work/outputs/week6_leakage_audit.csv

Error examples saved to:
/content/work/outputs/week6_error_examples.csv

SECTION 3 COMPLETE

Next:
SECTION 4 — CLAIM REWRITE


In [15]:
print("=" * 75)
print("SECTION 3 CONSISTENCY CHECK")
print("=" * 75)

print("comparison_df rows:", len(comparison_df))
print("X_test_grouped rows:", len(X_test_grouped))
print("y_test_grouped rows:", len(y_test_grouped))
print("test_df rows:", len(test_df))

error_total = len(test_df)

print("\nError-summary total:", error_summary["n"].sum())

print("\nExpected grouped test rows:",
      len(X_test_grouped))

if error_total != len(X_test_grouped):
    raise RuntimeError(
        f"""
❌ Section 3 consistency problem.

test_df contains {error_total} rows,
but X_test_grouped contains {len(X_test_grouped)} rows.

The failure examples are not aligned with the complete
Section-2 grouped test set.

Rerun Section 2 and then Section 3 in the same runtime.
"""
    )

if error_summary["n"].sum() != len(test_df):
    raise RuntimeError(
        "❌ Error categories do not cover all test rows."
    )

print("✅ Section 3 uses the complete grouped test set.")
print("✅ Error counts cover every grouped test row.")
print("✅ Section 3 is aligned with Section 2.")

SECTION 3 CONSISTENCY CHECK
comparison_df rows: 3292
X_test_grouped rows: 230
y_test_grouped rows: 230
test_df rows: 230

Error-summary total: 230

Expected grouped test rows: 230
✅ Section 3 uses the complete grouped test set.
✅ Error counts cover every grouped test row.
✅ Section 3 is aligned with Section 2.


# 4. Claim Rewrite — Keeping Conclusions Within the Evidence

## Purpose

The goal of this section is to make sure my conclusions do not go further than
the evidence produced by my Week-5 model and Week-6 validation audit.

My project is Lane 2: Refresh / Content Opportunity Scoring.

The model is intended to support prioritization of content for human review.
It is not intended to guarantee that a page will recover after a refresh.

## What I measured

My Week-6 honest validation used a client-grouped train/test split so that
clients in the test set were not present in the training set.

On the grouped test set:

- Week-4 baseline Average Precision: 0.5564
- Week-5 Logistic Regression Average Precision: 0.5754
- Random-split Logistic Regression Average Precision: 0.6085
- Change from random split to grouped split: -0.0331

The grouped result is the more conservative result because it evaluates the
model on clients that were held out from training.

## Claim 1 — Model performance

### Claim I would avoid

"The Logistic Regression model reliably predicts which pages will decline."

This is too strong because the experiment was performed on one validation
dataset and does not establish that the model will reliably predict future
declines in every client or future period.

### Safer claim

"On the grouped test set, the Logistic Regression model measured higher Average
Precision than the Week-4 baseline (0.5754 vs 0.5564)."

This is an observed and measured result from the evaluation performed in this
notebook.

### What the evidence supports

The evidence supports saying that the model showed a directional improvement
over the Week-4 baseline on the grouped test set.

It does not support claiming universal or production-level predictive
reliability.

---

## Claim 2 — Business/action impact

### Claim I would avoid

"Refreshing the pages selected by the model will improve their performance."

This is not supported because the model predicts an observed future decline
and ranks pages for review. The experiment does not test whether performing a
refresh causes recovery.

### Safer claim

"The model can be used as decision-support to prioritize pages for human
review based on their measured likelihood of future decline."

This describes what the model actually does without claiming a causal effect.

### What the evidence supports

The model provides a ranking signal that can help prioritize review candidates.

A human reviewer is still required to determine whether a refresh, rewrite,
protection, monitoring, or another action is appropriate.

---

## Validation lesson

The random split produced an Average Precision of 0.6085, while the
client-grouped split produced 0.5754.

The decrease of 0.0331 shows why validation design matters.

A random split can produce a more optimistic estimate when related pages from
the same client appear across training and testing.

The grouped split is therefore the more appropriate result for my current
client-level evaluation because the test clients were not seen during training.

## Final public-safe conclusion

My current evidence suggests that the Logistic Regression model provides a
directional improvement over the Week-4 baseline on the grouped test set.

The result should be treated as decision-support evidence rather than proof of
guaranteed future performance or proof that a content refresh will cause
recovery.

Further validation across additional time periods and unseen clients would be
needed before making a stronger production claim.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# WEEK 6 — SECTION 4: CLAIM REWRITE
# Lane 2: Refresh / Content Opportunity Scoring
# ============================================================

import os
import pandas as pd

print("=" * 75)
print("WEEK 6 — SECTION 4: CLAIM REWRITE")
print("=" * 75)


# ------------------------------------------------------------
# 1. Evidence values from the Week-6 grouped evaluation
# ------------------------------------------------------------

baseline_ap = 0.5564
random_split_ap = 0.6085
grouped_ap = 0.5754

validation_change = grouped_ap - random_split_ap
baseline_change = grouped_ap - baseline_ap


print("\n" + "=" * 75)
print("EVIDENCE USED FOR CLAIM REVIEW")
print("=" * 75)

print(f"Week-4 Baseline Average Precision : {baseline_ap:.4f}")
print(f"Random-Split Average Precision    : {random_split_ap:.4f}")
print(f"Grouped-Split Average Precision    : {grouped_ap:.4f}")

print(
    f"Grouped vs Random-Split Change     : "
    f"{validation_change:+.4f}"
)

print(
    f"Grouped Model vs Baseline Change   : "
    f"{baseline_change:+.4f}"
)


# ------------------------------------------------------------
# 2. Build the claim audit table
# ------------------------------------------------------------

claims = [
    {
        "claim_area": "Model performance",
        "unsafe_claim":
            "The Logistic Regression model reliably predicts which pages will decline.",
        "safe_claim":
            (
                "On the grouped test set, Logistic Regression measured "
                "higher Average Precision than the Week-4 baseline "
                "(0.5754 vs 0.5564)."
            ),
        "evidence_type": "Measured",
        "status": "REWRITTEN"
    },

    {
        "claim_area": "Business impact",
        "unsafe_claim":
            "Refreshing pages selected by the model will improve their performance.",
        "safe_claim":
            (
                "The model can be used as decision-support to prioritize "
                "pages for human review based on their measured likelihood "
                "of future decline."
            ),
        "evidence_type": "Decision-support",
        "status": "REWRITTEN"
    },

    {
        "claim_area": "Validation",
        "unsafe_claim":
            "The model has a 0.6085 Average Precision in general.",
        "safe_claim":
            (
                "Average Precision was 0.6085 under the random split and "
                "0.5754 under the client-grouped split."
            ),
        "evidence_type": "Observed",
        "status": "REWRITTEN"
    },

    {
        "claim_area": "Generalization",
        "unsafe_claim":
            "The model will perform this well for all clients.",
        "safe_claim":
            (
                "The grouped evaluation measured the model on clients "
                "held out from training; broader generalization would "
                "require additional validation periods and clients."
            ),
        "evidence_type": "Caveat",
        "status": "REWRITTEN"
    }
]


claims_df = pd.DataFrame(claims)


# ------------------------------------------------------------
# 3. Display the claim audit
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("CLAIM AUDIT TABLE")
print("=" * 75)

display(claims_df)


# ------------------------------------------------------------
# 4. Verify that unsafe causal language is not used
# ------------------------------------------------------------

unsafe_words = [
    "guarantee",
    "guaranteed",
    "prove",
    "proves",
    "causes",
    "caused",
    "will improve",
    "always",
    "reliably",
    "certain"
]

safe_text = " ".join(
    claims_df["safe_claim"].astype(str).str.lower()
)

found_unsafe_language = [
    word for word in unsafe_words
    if word in safe_text
]


print("\n" + "=" * 75)
print("PUBLIC-SAFE LANGUAGE CHECK")
print("=" * 75)

if found_unsafe_language:
    print("⚠️ Potentially unsafe language found:")
    print(found_unsafe_language)

else:
    print(
        "✅ No strong causal or guaranteed language detected "
        "in the rewritten claims."
    )


# ------------------------------------------------------------
# 5. Verify important evidence is present
# ------------------------------------------------------------

required_evidence = {
    "baseline_average_precision": baseline_ap,
    "random_split_average_precision": random_split_ap,
    "grouped_average_precision": grouped_ap,
    "grouped_minus_random": validation_change,
    "grouped_minus_baseline": baseline_change
}

print("\n" + "=" * 75)
print("EVIDENCE CHECK")
print("=" * 75)

for name, value in required_evidence.items():
    print(f"✅ {name}: {value:.4f}")


# ------------------------------------------------------------
# 6. Final interpretation
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL CLAIM INTERPRETATION")
print("=" * 75)

if grouped_ap > baseline_ap:
    print(
        "✅ The grouped model measured higher Average Precision "
        "than the Week-4 baseline."
    )
else:
    print(
        "⚠️ The grouped model did not exceed the Week-4 baseline."
    )


if grouped_ap < random_split_ap:
    print(
        "✅ Grouped validation produced a lower score than the "
        "random split, showing why validation design matters."
    )
else:
    print(
        "ℹ️ Grouped validation did not reduce the score relative "
        "to the random split."
    )


print(
    "\nThe result should be described as observed/measured "
    "decision-support evidence."
)

print(
    "It should NOT be described as proof that refreshing a page "
    "will cause recovery."
)


# ------------------------------------------------------------
# 7. Save the claim audit
# ------------------------------------------------------------

output_dir = "/content/work/outputs"
os.makedirs(output_dir, exist_ok=True)

claim_output_path = os.path.join(
    output_dir,
    "week6_claim_rewrite.csv"
)

claims_df.to_csv(
    claim_output_path,
    index=False
)

print("\n" + "=" * 75)
print("SECTION 4 COMPLETE")
print("=" * 75)

print("Claim audit saved to:")
print(claim_output_path)

print("\nNext:")
print("SECTION 5 — SELF-CHECK")

WEEK 6 — SECTION 4: CLAIM REWRITE

EVIDENCE USED FOR CLAIM REVIEW
Week-4 Baseline Average Precision : 0.5564
Random-Split Average Precision    : 0.6085
Grouped-Split Average Precision    : 0.5754
Grouped vs Random-Split Change     : -0.0331
Grouped Model vs Baseline Change   : +0.0190

CLAIM AUDIT TABLE


,claim_area,unsafe_claim,safe_claim,evidence_type,status
0,Model performance,The Logistic Regression model reliably predict...,"On the grouped test set, Logistic Regression m...",Measured,REWRITTEN
1,Business impact,Refreshing pages selected by the model will im...,The model can be used as decision-support to p...,Decision-support,REWRITTEN
2,Validation,The model has a 0.6085 Average Precision in ge...,Average Precision was 0.6085 under the random ...,Observed,REWRITTEN
3,Generalization,The model will perform this well for all clients.,The grouped evaluation measured the model on c...,Caveat,REWRITTEN



PUBLIC-SAFE LANGUAGE CHECK
✅ No strong causal or guaranteed language detected in the rewritten claims.

EVIDENCE CHECK
✅ baseline_average_precision: 0.5564
✅ random_split_average_precision: 0.6085
✅ grouped_average_precision: 0.5754
✅ grouped_minus_random: -0.0331
✅ grouped_minus_baseline: 0.0190

FINAL CLAIM INTERPRETATION
✅ The grouped model measured higher Average Precision than the Week-4 baseline.
✅ Grouped validation produced a lower score than the random split, showing why validation design matters.

The result should be described as observed/measured decision-support evidence.
It should NOT be described as proof that refreshing a page will cause recovery.

SECTION 4 COMPLETE
Claim audit saved to:
/content/work/outputs/week6_claim_rewrite.csv

Next:
SECTION 5 — SELF-CHECK


# 5. Self-Check

This final self-check verifies that the Week-6 validation audit is internally
consistent, leakage-controlled, reproducible, and described using claims that
match the evidence.

## Problem and decision

- Lane: Lane 2 — Refresh / Content Opportunity Scoring
- Decision: which content should be prioritized for human review
- Unit of analysis: one client × content item
- Feature window: March 2026
- Observed outcome window: April 2026
- Target: `future_decline`
- Model: Logistic Regression
- Baseline: Week-4 `baseline_score`

## Validation

The Week-5 model was evaluated using both a conventional random split and a
client-grouped split.

The client-grouped split was used as the more honest validation design because
clients can have shared patterns across their content. The grouped test set
contained clients that were not present in the training set, and the client
overlap check returned zero.

Observed Average Precision:

- Random split: 0.6085
- Client-grouped split: 0.5754
- Week-4 baseline on the grouped test set: 0.5564

The grouped model therefore measured an absolute Average Precision difference
of +0.0190 relative to the Week-4 baseline on this test set.

The random-to-grouped change was -0.0331, showing that the validation design
materially affects the measured result.

## Leakage

The model used only:

- `march_impressions`
- `march_clicks`
- `march_avg_position`

April variables, `future_decline`, `click_change_pct`, and Week-4 decision
outputs were not used as model features.

The leakage audit and model-matrix checks passed.

## Errors

The grouped test set was checked for false positives and false negatives.
These examples show that the model does not perfectly separate future declines
from non-declines.

The errors are treated as evidence for model limitations rather than as proof
that the model is wrong in every similar case.

## Claim discipline

I will describe the result as:

- observed;
- measured;
- directional;
- decision-support.

I will not claim that:

- the model guarantees future performance;
- the model works equally well for every client or future period;
- refreshing a page causes recovery;
- the model proves a causal relationship.

## Final assessment

The Week-6 audit supports the conclusion that the Logistic Regression model
showed a measured directional improvement over the Week-4 baseline on the
grouped test set.

The result is not a universal production-performance claim. Additional
validation across more clients and future time periods would be required before
making stronger generalization claims.

This notebook is considered complete only after all previous sections have
been executed successfully and the final outputs have been checked.

In [17]:
# ============================================================
# WEEK 6 — SECTION 5
# FINAL SELF-CHECK
# ============================================================

import os
import pandas as pd

print("=" * 75)
print("WEEK 6 — FINAL SELF-CHECK")
print("=" * 75)


# ============================================================
# 1. Check required Section-2 objects
# ============================================================

required_section2_objects = [
    "comparison_df",
    "grouped_model",
    "grouped_probability",
    "grouped_prediction",
    "test_idx",
    "X_test_grouped",
    "y_test_grouped"
]

missing_section2 = [
    obj
    for obj in required_section2_objects
    if obj not in globals()
]

if missing_section2:
    raise RuntimeError(
        "Missing Section-2 objects: "
        + str(missing_section2)
    )

print("✅ Section-2 objects available.")


# ============================================================
# 2. Check Section-3 objects
# ============================================================

required_section3_objects = [
    "leakage_audit_df",
    "test_df",
    "error_summary"
]

missing_section3 = [
    obj
    for obj in required_section3_objects
    if obj not in globals()
]

if missing_section3:
    raise RuntimeError(
        "Missing Section-3 objects: "
        + str(missing_section3)
    )

print("✅ Section-3 audit objects available.")


# ============================================================
# 3. Check Section-4 objects
# ============================================================

required_section4_objects = [
    "claims_df"
]

missing_section4 = [
    obj
    for obj in required_section4_objects
    if obj not in globals()
]

if missing_section4:
    raise RuntimeError(
        "Missing Section-4 objects: "
        + str(missing_section4)
    )

print("✅ Section-4 claim audit available.")


# ============================================================
# 4. Check model features
# ============================================================

expected_features = [
    "march_impressions",
    "march_clicks",
    "march_avg_position"
]

actual_features = list(
    X_test_grouped.columns
)

print("\n" + "=" * 75)
print("MODEL FEATURE CHECK")
print("=" * 75)

print("Expected:")
print(expected_features)

print("\nActual:")
print(actual_features)

if actual_features != expected_features:
    raise RuntimeError(
        "Model feature list does not exactly match "
        "the approved March feature set."
    )

print("✅ Model uses only approved March features.")


# ============================================================
# 5. Check forbidden features
# ============================================================

forbidden_features = [
    "april_impressions",
    "april_clicks",
    "april_avg_position",
    "click_change_pct",
    "future_decline",
    "baseline_score",
    "reason_code",
    "action_label"
]

forbidden_used = [
    col
    for col in actual_features
    if col in forbidden_features
]

if forbidden_used:
    raise RuntimeError(
        "Potential leakage detected: "
        + str(forbidden_used)
    )

print("✅ No forbidden feature is in the model matrix.")


# ============================================================
# 6. Check grouped validation
# ============================================================

train_clients = set(
    comparison_df.iloc[test_idx]["client_hash_id"].unique()
)

# Reconstruct actual train clients from the grouped split
all_indices = set(range(len(comparison_df)))
test_indices = set(test_idx)

train_indices = sorted(
    all_indices - test_indices
)

actual_train_clients = set(
    comparison_df.iloc[train_indices]["client_hash_id"].unique()
)

actual_test_clients = set(
    comparison_df.iloc[test_idx]["client_hash_id"].unique()
)

overlap = (
    actual_train_clients
    .intersection(actual_test_clients)
)

print("\n" + "=" * 75)
print("GROUPED VALIDATION CHECK")
print("=" * 75)

print(
    "Training clients:",
    len(actual_train_clients)
)

print(
    "Testing clients:",
    len(actual_test_clients)
)

print(
    "Client overlap:",
    len(overlap)
)

if len(overlap) != 0:
    raise RuntimeError(
        "Client overlap detected."
    )

print("✅ No client appears in both train and test.")


# ============================================================
# 7. Check error coverage
# ============================================================

error_rows = (
    error_summary["n"].sum()
)

test_rows = len(test_df)

print("\n" + "=" * 75)
print("ERROR COVERAGE CHECK")
print("=" * 75)

print("Grouped test rows:", test_rows)
print("Error-summary rows:", error_rows)

if error_rows != test_rows:
    raise RuntimeError(
        "Error categories do not cover the complete "
        "grouped test set."
    )

print("✅ Every grouped test row has an error category.")


# ============================================================
# 8. Check the measured metrics
# ============================================================

baseline_ap = 0.5564
random_ap = 0.6085
grouped_ap = 0.5754

grouped_vs_baseline = (
    grouped_ap - baseline_ap
)

random_vs_grouped = (
    grouped_ap - random_ap
)

print("\n" + "=" * 75)
print("METRIC CHECK")
print("=" * 75)

print(
    f"Week-4 baseline AP : {baseline_ap:.4f}"
)

print(
    f"Random split AP    : {random_ap:.4f}"
)

print(
    f"Grouped split AP   : {grouped_ap:.4f}"
)

print(
    f"Grouped - baseline : {grouped_vs_baseline:+.4f}"
)

print(
    f"Grouped - random   : {random_vs_grouped:+.4f}"
)

if grouped_ap > baseline_ap:
    print(
        "✅ Grouped model measured higher AP than baseline."
    )
else:
    print(
        "⚠️ Grouped model did not exceed baseline."
    )


# ============================================================
# 9. Check public-safe claims
# ============================================================

unsafe_terms = [
    "guarantee",
    "guaranteed",
    "proves",
    "prove",
    "causes",
    "caused",
    "always"
]

claim_text = (
    claims_df["safe_claim"]
    .astype(str)
    .str.lower()
    .str.cat(sep=" ")
)

unsafe_found = [
    term
    for term in unsafe_terms
    if term in claim_text
]

print("\n" + "=" * 75)
print("PUBLIC-SAFE CLAIM CHECK")
print("=" * 75)

if unsafe_found:
    print(
        "⚠️ Review these terms:",
        unsafe_found
    )
else:
    print(
        "✅ No prohibited strong causal/guarantee terms "
        "found in safe claims."
    )


# ============================================================
# 10. Check output files
# ============================================================

expected_outputs = [
    "/content/work/outputs/week6_validation_comparison.csv",
    "/content/work/outputs/week6_leakage_audit.csv",
    "/content/work/outputs/week6_error_examples.csv",
    "/content/work/outputs/week6_claim_rewrite.csv"
]

print("\n" + "=" * 75)
print("OUTPUT FILE CHECK")
print("=" * 75)

missing_outputs = []

for path in expected_outputs:

    if os.path.exists(path):
        print("✅", path)
    else:
        print("❌", path)
        missing_outputs.append(path)


if missing_outputs:
    raise FileNotFoundError(
        "Missing expected output files: "
        + str(missing_outputs)
    )


# ============================================================
# 11. Final checklist
# ============================================================

final_checks = {
    "Problem framed as decision-support": True,
    "March feature window defined": True,
    "April observed outcome defined": True,
    "Client-grouped validation performed": True,
    "No train/test client overlap": len(overlap) == 0,
    "Only March features used": actual_features == expected_features,
    "No future feature leakage": len(forbidden_used) == 0,
    "All grouped test rows have error labels":
        error_rows == test_rows,
    "Baseline compared on grouped test set": True,
    "False positives inspected": True,
    "False negatives inspected": True,
    "Claims rewritten conservatively": len(unsafe_found) == 0,
    "Required output files created": len(missing_outputs) == 0
}


print("\n" + "=" * 75)
print("FINAL WEEK-6 CHECKLIST")
print("=" * 75)

for check, passed in final_checks.items():

    symbol = "✅" if passed else "❌"

    print(
        f"{symbol} {check}"
    )


# ============================================================
# 12. Final status
# ============================================================

all_passed = all(
    final_checks.values()
)

print("\n" + "=" * 75)

if all_passed:

    print("✅ WEEK 6 SELF-CHECK PASSED")
    print("=" * 75)

    print(
        "\nThe notebook is ready for final review."
    )

    print(
        "\nBefore committing:"
        "\n1. Run all cells from top to bottom."
        "\n2. Confirm there are no errors."
        "\n3. Confirm no secrets/tokens are present."
        "\n4. Save the executed notebook as:"
        "\n   work/notebooks/w06_validation_audit.ipynb"
        "\n5. Commit and push the notebook to GitHub."
    )

else:

    print("❌ WEEK 6 SELF-CHECK FAILED")
    print("=" * 75)

    print(
        "\nFix the failed checks before submitting."
    )

WEEK 6 — FINAL SELF-CHECK
✅ Section-2 objects available.
✅ Section-3 audit objects available.
✅ Section-4 claim audit available.

MODEL FEATURE CHECK
Expected:
['march_impressions', 'march_clicks', 'march_avg_position']

Actual:
['march_impressions', 'march_clicks', 'march_avg_position']
✅ Model uses only approved March features.
✅ No forbidden feature is in the model matrix.

GROUPED VALIDATION CHECK
Training clients: 15
Testing clients: 7
Client overlap: 0
✅ No client appears in both train and test.

ERROR COVERAGE CHECK
Grouped test rows: 230
Error-summary rows: 230
✅ Every grouped test row has an error category.

METRIC CHECK
Week-4 baseline AP : 0.5564
Random split AP    : 0.6085
Grouped split AP   : 0.5754
Grouped - baseline : +0.0190
Grouped - random   : -0.0331
✅ Grouped model measured higher AP than baseline.

PUBLIC-SAFE CLAIM CHECK
✅ No prohibited strong causal/guarantee terms found in safe claims.

OUTPUT FILE CHECK
✅ /content/work/outputs/week6_validation_comparison.csv
✅ 